## **OBJECTIF of this notebook: I'm going to implement a offline lightweigth model yo use in my Koobo app.**

In [ ]:
import tensorflow as tf

In [ ]:
IMG=224
BATCH=32

In [ ]:
# Splitting the dataset into
train = tf.keras.utils.image_dataset_from_directory("data/train", image_size=(IMG,IMG), batch_size=BATCH)
val   = tf.keras.utils.image_dataset_from_directory("data/val",   image_size=(IMG,IMG), batch_size=BATCH)
classes = train.class_names

aug = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.15),
  tf.keras.layers.RandomZoom(0.15),
  tf.keras.layers.RandomBrightness(0.2),
])
base = tf.keras.applications.MobileNetV3Small(input_shape=(IMG,IMG,3), include_top=False, weights="imagenet")
base.trainable = False  # phase 1 : on gèle la base

x = tf.keras.Input((IMG,IMG,3))
y = aug(x)
y = tf.keras.applications.mobilenet_v3.preprocess_input(y)
y = base(y, training=False)
y = tf.keras.layers.GlobalAveragePooling2D()(y)
y = tf.keras.layers.Dropout(0.3)(y)
y = tf.keras.layers.Dense(len(classes), activation="softmax")(y)
model = tf.keras.Model(x, y)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=10)

# phase 2 (fine-tuning) : dégeler les dernières couches, lr faible
base.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=5)
model.save("koobo_disease.keras")

FileNotFoundError: [Errno 2] No such file or directory: 'data/train'